# 🏦 Loan Approval Prediction — Exploratory Data Analysis

> **Goal:** Understand the dataset, uncover patterns, and identify the key features that drive loan approval decisions before building any model.

---

## 📋 Project Overview

| Item | Detail |
|---|---|
| **Dataset** | Combined Loan Dataset |
| **Records** | 614 applicants |
| **Features** | 12 input features + 1 target |
| **Target** | `Loan_Status` (Y = Approved, N = Rejected) |
| **Problem Type** | Binary Classification |
| **Live App** | Deployed on Render — [Loan Approval Predictor](https://loan-approval-using-ml.onrender.com) |

---

## 📑 Table of Contents
1. [Imports & Setup](#1)
2. [Load Dataset](#2)
3. [Dataset Overview](#3)
4. [Missing Value Analysis](#4)
5. [Target Variable Distribution](#5)
6. [Univariate Analysis — Categorical Features](#6)
7. [Univariate Analysis — Numerical Features](#7)
8. [Bivariate Analysis — Feature vs Loan Status](#8)
9. [Correlation Analysis](#9)
10. [EDA Summary & Key Insights](#10)


## 1. Imports & Setup <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print("Libraries loaded ✅")


## 2. Load Dataset <a id='2'></a>

We load the combined loan dataset and immediately filter out rows where `Loan_Status` is missing, since those cannot be used for supervised learning.


In [ ]:
df = pd.read_csv('combined_loan_dataset.csv')
df = df[df['Loan_Status'].notna()]

print(f"Dataset shape: {df.shape}")
df.sample(10)


## 3. Dataset Overview <a id='3'></a>

Let's inspect the column types, non-null counts, and basic statistics to understand what we're working with.


In [ ]:
print("=== Data Types & Non-Null Counts ===")
df.info()


In [ ]:
print("=== Descriptive Statistics ===")
df.describe().T.style.background_gradient(cmap='Blues')


**Observations:**
- `ApplicantIncome` and `CoapplicantIncome` are heavily right-skewed (mean >> median implied by large std).
- `LoanAmount` also shows skew and likely contains outliers.
- `Credit_History` is binary (0 or 1) — it's categorical in nature despite being stored as float.
- `Loan_Amount_Term` is mostly 360 months (30-year loans).


## 4. Missing Value Analysis <a id='4'></a>

Understanding missingness is critical before any preprocessing step.


In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Missing Values by Feature (%)')
ax.set_xlabel('Feature')
ax.set_ylabel('Missing %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x()+p.get_width()/2, p.get_height()+0.1),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()


**Key takeaways:**
- `Credit_History` has the highest missingness (~8%) — it will be imputed with the most frequent value.
- All missing rates are below 10%, so we can impute safely rather than drop rows.
- We'll use `SimpleImputer(strategy='most_frequent')` for categoricals and `strategy='median'` for numericals.


## 5. Target Variable Distribution <a id='5'></a>

Before EDA, we check for class imbalance — this directly affects model training strategy.


In [ ]:
counts = df['Loan_Status'].value_counts()
pcts   = df['Loan_Status'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar
axes[0].bar(counts.index, counts.values, color=['#4f8ef7','#ef4444'], edgecolor='white', width=0.5)
axes[0].set_title('Loan Status — Count')
axes[0].set_xlabel('Loan Status')
axes[0].set_ylabel('Count')
for i, (v, p) in enumerate(zip(counts.values, pcts.values)):
    axes[0].text(i, v + 3, f'{v}\n({p:.1f}%)', ha='center', fontsize=10)

# Pie
axes[1].pie(counts.values, labels=['Approved (Y)', 'Rejected (N)'],
            colors=['#4f8ef7','#ef4444'], autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Loan Status — Proportion')

plt.suptitle('Target Variable: Loan Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"\nClass ratio  →  Approved: {pcts['Y']:.1f}%  |  Rejected: {pcts['N']:.1f}%")
print("⚠️  Dataset is moderately imbalanced — RandomOverSampler will be applied during preprocessing.")


## 6. Univariate Analysis — Categorical Features <a id='6'></a>

We examine the distribution of each categorical feature to understand the applicant profile in this dataset.


In [ ]:
cat_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    counts = df[col].value_counts()
    bars = axes[i].bar(counts.index, counts.values,
                       color=sns.color_palette('muted', len(counts)),
                       edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2, h + 1,
                     str(h), ha='center', va='bottom', fontsize=9)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Insights:**
- **Gender:** ~80% male applicants — the dataset skews male.
- **Married:** Majority of applicants are married.
- **Dependents:** Most applicants have 0 dependents.
- **Education:** Graduates dominate (~78%).
- **Self_Employed:** ~86% are not self-employed.
- **Property_Area:** Semiurban has the most applicants, followed by Urban and Rural.


## 7. Univariate Analysis — Numerical Features <a id='7'></a>

We use KDE plots and boxplots together to understand the distribution shape and detect outliers.


In [ ]:
num_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(num_features):
    # KDE
    sns.kdeplot(df[col].dropna(), ax=axes[0, i], fill=True, color='steelblue')
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel('')
    # Boxplot
    axes[1, i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, i].set_title(f'{col} — Boxplot')
    axes[1, i].set_xlabel('')

plt.suptitle('Numerical Feature Distributions & Outliers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Insights:**
- **ApplicantIncome:** Strongly right-skewed. A small group earns very high incomes — these are outliers.
- **CoapplicantIncome:** Heavily skewed — many applicants have a co-applicant with 0 income.
- **LoanAmount:** Right-skewed with clear outliers. Yeo-Johnson transform will be applied for Logistic Regression.
- **Loan_Amount_Term:** Most loans are 360 months (30 years). Very few outliers.


## 8. Bivariate Analysis — Feature vs Loan Status <a id='8'></a>

Here we examine how each feature relates to the loan approval outcome.


### 8a. Categorical Features vs Loan Status

In [ ]:
cat_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Credit_History']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df['Loan_Status'])
    ct.plot(kind='bar', ax=axes[i], color=['#ef4444','#22c55e'], edgecolor='white', rot=30)
    axes[i].set_title(f'{col} vs Loan Status')
    axes[i].set_xlabel('')
    axes[i].legend(['Rejected (N)', 'Approved (Y)'], fontsize=8)
    axes[i].tick_params(axis='x', labelsize=8)

# Hide unused subplot
axes[-1].set_visible(False)
plt.suptitle('Categorical Features vs Loan Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Key findings:**
- **Credit_History** is the single most powerful predictor — applicants with Credit_History=1 are approved at a much higher rate.
- **Education:** Graduates have higher approval rates than non-graduates.
- **Married:** Married applicants are more likely to be approved.
- **Property_Area:** Semiurban applicants show the highest approval rate.
- **Gender & Self_Employed:** Relatively small effect on approval.


### 8b. Numerical Features vs Loan Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Income vs LoanAmount coloured by approval
sns.scatterplot(x='ApplicantIncome', y='LoanAmount', hue='Loan_Status',
                data=df, palette={'Y': '#22c55e', 'N': '#ef4444'},
                alpha=0.6, ax=axes[0])
axes[0].set_title('Applicant Income vs Loan Amount\n(coloured by Loan Status)')
axes[0].set_xlabel('Applicant Income ($)')
axes[0].set_ylabel('Loan Amount ($K)')
axes[0].legend(title='Loan Status', labels=['Approved', 'Rejected'])

# Boxplot: Income distribution by Loan Status
df.boxplot(column='ApplicantIncome', by='Loan_Status', ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Applicant Income by Loan Status')
axes[1].set_xlabel('Loan Status')
axes[1].set_ylabel('Applicant Income ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()


## 9. Correlation Analysis <a id='9'></a>

We examine linear correlations between numerical features and the encoded target.


In [ ]:
df_corr = df.copy()
df_corr['Loan_Status_encoded'] = df_corr['Loan_Status'].map({'Y': 1, 'N': 0})

corr_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
             'Loan_Amount_Term', 'Credit_History', 'Loan_Status_encoded']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            vmin=-1, vmax=1, center=0,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numerical Features', fontweight='bold')
plt.tight_layout()
plt.show()


**Correlation insights:**
- `ApplicantIncome` ↔ `LoanAmount`: **+0.57** — the strongest feature-feature correlation. Higher income → higher loan requested. Makes intuitive sense.
- `CoapplicantIncome` ↔ `LoanAmount`: **+0.19** — weak positive correlation.
- `Credit_History` ↔ `Loan_Status`: Highest correlation with the target. Confirms it is the most important feature.
- Most numerical features have very weak linear correlation with Loan_Status, suggesting non-linear models (RF, GB) may outperform Logistic Regression alone.


## 10. EDA Summary & Key Insights <a id='10'></a>

---

### 📊 Dataset Overview
| Property | Value |
|---|---|
| Total records | 614 |
| Features | 12 input + 1 target |
| Missing values | Yes (< 10% per column) |
| Class imbalance | ~69% Approved / ~31% Rejected |

---

### 🔑 Top Findings

| Rank | Feature | Insight |
|---|---|---|
| 1 | **Credit History** | Strongest single predictor. Credit_History=1 leads to significantly higher approval rates. |
| 2 | **ApplicantIncome** | Right-skewed. Positively correlated with LoanAmount (~0.57). |
| 3 | **Property Area** | Semiurban applicants have the highest approval rate. |
| 4 | **Education** | Graduates are approved more often. |
| 5 | **Married** | Married applicants show higher approval rates. |

---

### ⚙️ Preprocessing Decisions (informed by EDA)

| Decision | Reason |
|---|---|
| Impute missing with `most_frequent` | Categoricals have < 10% missing |
| Impute missing with `median` | Numerical features are skewed — median is robust |
| `PowerTransformer (Yeo-Johnson)` for LR | Logistic Regression assumes near-normal inputs |
| `RandomOverSampler` | 69/31 imbalance — oversampling minority class improves recall |
| `OneHotEncoder` for categoricals | No ordinal relationship in most categorical features |

---

> ➡️ **Next:** See the Preprocessing & Modelling notebook for the full pipeline, GridSearchCV tuning, and the final VotingClassifier ensemble.
